# Day 20 — Multi-Project Optimizer Test

**Goal:** Run the optimizer across 5 projects simultaneously and
verify no employee is double-booked and all staffable roles are filled.

In [1]:
import pandas as pd

plan = pd.read_csv("../data/processed/staffing_plan.csv")
score_matrix = pd.read_csv("../data/processed/score_matrix.csv")

print(f"Total assignments: {len(plan)}")
plan

Total assignments: 14


,project_id,role,employee_id,final_score
0,P001,Backend Dev,E061,0.8389
1,P001,Data Engineer,E031,0.7473
2,P001,DevOps,E041,0.8451
3,P002,Android Dev,E037,0.8307
4,P002,Backend Dev,E066,0.8676
5,P002,Data Engineer,E058,0.8118
6,P003,Android Dev,E068,0.7433
7,P003,Backend Dev,E075,0.8688
8,P003,Data Engineer,E016,0.7438
9,P004,Data Engineer,E064,0.8027


## Check 1 — No employee double-booked

In [2]:
dupes = plan[plan.duplicated("employee_id", keep=False)]
print(f"Double-booked employees: {len(dupes)}")
assert len(dupes) == 0, "Found double-booking — constraint failed!"
print("✅ No double-booking")

Double-booked employees: 0
✅ No double-booking


## Check 2 — Every staffable (project, role) slot filled

In [3]:
all_slots = (score_matrix[score_matrix.eligible == True]
             [["project_id", "role"]]
             .drop_duplicates())

filled_slots = plan[["project_id", "role"]].drop_duplicates()

missing = all_slots.merge(filled_slots, on=["project_id", "role"],
                           how="left", indicator=True)
missing = missing[missing._merge == "left_only"]

print(f"Unfilled slots: {len(missing)}")
if len(missing):
    print(missing[["project_id", "role"]])
else:
    print("✅ All staffable slots filled")

Unfilled slots: 0
✅ All staffable slots filled


## Check 3 — Overall plan quality

In [4]:
print(f"Average match score : {plan.final_score.mean():.3f}")
print(f"Total objective     : {plan.final_score.sum():.3f}")
print(f"\nAssignments per project:")
print(plan.groupby("project_id").size().rename("roles_filled"))

Average match score : 0.810
Total objective     : 11.345

Assignments per project:
project_id
P001    3
P002    3
P003    3
P004    3
P005    2
Name: roles_filled, dtype: int64


# Day 21 — Edge Cases: Unstaffable Roles & Infeasibility

Two failure modes handled in `optimize_staffing.py`:

1. **A role has zero eligible candidates** → `find_unstaffable_roles()`
   catches this *before* solving so you get a clear warning instead
   of a silently unfilled slot.
2. **The whole model is infeasible** (too many roles, too few people)
   → `solve()` detects `cp_model.INFEASIBLE` and returns an empty
   DataFrame with a clear message instead of crashing.

## Test 1 — Check for unstaffable roles across full dataset

In [5]:
import sys
sys.path.append("../src")
from optimize_staffing import StaffingOptimizer

opt_test = StaffingOptimizer(
    score_matrix_path="../data/processed/score_matrix.csv",
    employees_path="../data/processed/employees_with_index.csv",
)

unstaffable = opt_test.find_unstaffable_roles()
print(f"Unstaffable roles across full dataset: {len(unstaffable)}")
if len(unstaffable):
    print(unstaffable.to_string(index=False))
else:
    print("✅ Every (project, role) slot has at least one eligible candidate")

Unstaffable roles across full dataset: 0
✅ Every (project, role) slot has at least one eligible candidate


## Test 2 — Deliberately trigger infeasibility

Force infeasibility by asking the solver to staff more roles than
there are eligible employees — we limit to only 2 employees via a
temporarily filtered score matrix written inline.

In [6]:
from ortools.sat.python import cp_model
import pandas as pd

# Build a tiny infeasible problem: 3 roles but only 2 people
scores_tiny = pd.DataFrame([
    {"project_id": "PX", "role": "RoleA", "employee_id": "E001", "final_score": 0.9, "eligible": True},
    {"project_id": "PX", "role": "RoleB", "employee_id": "E001", "final_score": 0.8, "eligible": True},
    {"project_id": "PX", "role": "RoleC", "employee_id": "E001", "final_score": 0.7, "eligible": True},
    {"project_id": "PX", "role": "RoleA", "employee_id": "E002", "final_score": 0.6, "eligible": True},
    {"project_id": "PX", "role": "RoleB", "employee_id": "E002", "final_score": 0.5, "eligible": True},
    {"project_id": "PX", "role": "RoleC", "employee_id": "E002", "final_score": 0.4, "eligible": True},
])

# 3 roles, only 2 people → one role can't be filled → INFEASIBLE
model = cp_model.CpModel()
x = {}
for _, row in scores_tiny.iterrows():
    key = (row["employee_id"], row["project_id"], row["role"])
    x[key] = model.NewBoolVar(f"x_{key[0]}_{key[1]}_{key[2]}")

for (pid, role), group in scores_tiny.groupby(["project_id", "role"]):
    model.Add(sum(x[(r["employee_id"], pid, role)] for _, r in group.iterrows()) == 1)

for emp_id, group in scores_tiny.groupby("employee_id"):
    model.Add(sum(x[(emp_id, r["project_id"], r["role"])] for _, r in group.iterrows()) <= 1)

model.Maximize(sum(int(row["final_score"] * 10000) * x[(row["employee_id"], row["project_id"], row["role"])]
                    for _, row in scores_tiny.iterrows()))

solver = cp_model.CpSolver()
status = solver.Solve(model)

print(f"Status: {solver.StatusName(status)}")
if status == cp_model.INFEASIBLE:
    print("✅ Infeasibility correctly detected — 3 roles, only 2 people available")
    print("   In production: check find_unstaffable_roles() or increase eligible pool")

Status: INFEASIBLE
✅ Infeasibility correctly detected — 3 roles, only 2 people available
   In production: check find_unstaffable_roles() or increase eligible pool


## ✅ Week 3 Sprint Retro

**Sprint Goal ACHIEVED:** Solver turns match scores into a conflict-free,
constraint-respecting staffing plan ✅

| Day   | What we built                                              |
|-------|------------------------------------------------------------|
| 15    | Assignment problem theory, worked examples by hand         |
| 16    | OR-Tools installed, toy CP-SAT script verified vs brute force |
| 17    | Written problem formulation (vars, objective, constraints) |
| 18-19 | optimize_staffing.py — full CP-SAT model, P001+P002 test  |
| 20    | Multi-project test (5 projects) — no double-booking ✅     |
| 21    | Unstaffable-role detection, infeasibility handling ✅       |

**Plan quality:** 14 roles filled across 5 projects, avg score 0.810,
zero double-bookings, solver reached OPTIMAL on all runs.

**Next → Week 4:** RAG + LLM explanation layer + Streamlit dashboard